# Taylor-SWFT: How to use the code

*Authors:* Marius RODRIGUES, Louis LALAY

*Contact:* marius.rodrigues@telecom-paris.fr

*Last updated:* 2026, April 27

*Tested on:* Ubuntu 24.04.4 (CPU Intel® Core™ Ultra 9 285 × 24, RAM 32Go)

**Notebook outline:**
1. [*Create a `SWFTRoom`*](#1---create-a-swftroom) (How to build a room object and compute the reverberation parameters)
    - [*Method 1 (simple): use the `from_corners()` class method*](#method-1-simple-use-the-from_corners-class-method)
    - [*Method 2 (flexible): Instanciate from `pyroomacoustics.Room`*](#method-2-flexible-instanciate-from-pyroomacousticsroom)
2. [*Reverberator*](#2---reverberator) (The base object for RIR simulation)
3. [*Real-Time Processing*](#3---real-time-processing) (The high and low-level APIs for real-time dynamic reverberation)
    - [*High-level usage*](#high-level-usage)
    - [*Low-level usage*](#low-level-usage)

## 0 - Import the necessary packages for the notebook

In [ ]:
from IPython.display import Audio
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from pprint import pprint
from time import perf_counter
from torchaudio.functional import convolve
import numpy as np
import pyroomacoustics as pra
import taylor_swft as ts
import torch
import trimesh as tm

EXAMPLE_PATH = Path("data/examples/")

## 1 - Create a `SWFTRoom`

`SWFTRoom` is the base class that allows to compute all acoustic parameters from the room's shape and material. It can be instanciated in two different ways. `SWFTRoom` handles  from the Statistical Wave Field Theory

### Method 1 (simple): use the `from_corners()` class method

This method build the SWFTRoom as an extruded polyon, whose corners are given in the counter-clockwise order.

In [ ]:
base_corners = np.array(
    [[0, 0], [4, 0], [3, 7], [-2, 7], [-4, 6], [-2, 4], [1, 4], [1, 2], [-1, 2]]
)

To instanciate the room, it is also needed to define materials and associate them to each wall. The material absorptions should be given per octave band, with center frequencies starting from $f_{\text{base}} = 125 Hz$. Note that the number $N$ of coefficients provided automatically defines the frequency sampling as $F_s = f_{\text{base}}\times 2^{N}$.
Material absorptions are then associated to each wall in a dictonnary using the following convention:
```
absorption_dict = {
    "list_walls" : [wall_0_material, wall_1_material, ...], ### wall_{i} goes from corner_{i} to corner_{(i+1)%N_corners}
    "ceiling" : ceiling_material,
    "floor" : floor_material
}
```

The current implementation of Taylor-SWFT also supports materials defined from their (complex) specific admittances. The relationship between absorptions $a(f)$ and admittances $\hat\beta(f)$ is:
$$\ln(1 - a(f))= 2\int_0^1 \ln\left(\left|\frac{u - \hat{\beta}(f)}{u + \hat{\beta}(f)}\right|^2\right)u \text{d}u$$

In [ ]:
# material absorptions below were taken from the pyroomacoustics database : https://pyroomacoustics.readthedocs.io/en/pypi-release/pyroomacoustics.materials.database.html
# center frequencies : 125 Hz, 250 Hz, 500 Hz, 1 kHz, 2 kHz, 4 kHz, 8 kHz
rough_concrete = [0.02, 0.03, 0.03, 0.03, 0.04, 0.07, 0.07]
carpet_cotton = [0.07, 0.31, 0.49, 0.81, 0.66, 0.54, 0.48]
curtains_02 = [0.05, 0.06, 0.39, 0.63, 0.7, 0.73, 0.73]
ceiling_plaster_board = [
    0.2,
    0.15,
    0.1,
    0.08,
    0.04,
    0.02,
    0.02,
]  # 8kHz value is fixed equal to the 4kHz value, as it was not provided in the database
absorptions_dict = {
    "list_walls": [
        rough_concrete,
        rough_concrete,
        curtains_02,
        curtains_02,
        curtains_02,
        curtains_02,
        rough_concrete,
        rough_concrete,
        rough_concrete,
    ],
    "ceiling": ceiling_plaster_board,
    "floor": carpet_cotton,
}

A `SWFTRoom` can then be instaciated using the `from_corners()` method as below. Admittances can be provided instead of absorptions, through the `admittances` keyword argument.

In [ ]:
swft_room = ts.SWFTRoom.from_corners(
    base_corners, height=3.0, absorptions=absorptions_dict
)
swft_room.room.plot()
plt.show()

### Method 2 (flexible): Instanciate from `pyroomacoustics.Room`

#### Step 1: Define some materials in the pyroomacoustics standards

The pyroomacoustics library provides a framework to define new materials under the `Material` class. Using this framework to define the walls absorptions (and potential scattering coefficients) before instanciating a SWFTRoom is not strictly necessary but strongly advised to ensure parameters consistency.

In [ ]:
material_dict = {
    "rough_concrete": rough_concrete,
    "carpet_cotton": carpet_cotton,
    "curtains_02": curtains_02,
    "ceiling_plaster_board": ceiling_plaster_board,
}
center_freqs = [125, 250, 500, 1000, 2000, 4000, 8000]

# we create a dictonnary of pyroomacoustics materials, which will be used to create the walls of the pyroomacoustics room
pra_material_library: dict[str, pra.parameters.Material] = {}
for material_name, absorption in material_dict.items():
    material = pra.parameters.Material(
        energy_absorption={
            "description": material_name,
            "coeffs": absorption,
            "center_freqs": center_freqs,
        },
        scattering={
            "description": material_name,
            "coeffs": [0.0] * len(absorption),
            "center_freqs": center_freqs,
        },
    )
    pra_material_library[material_name] = material

#### Step 2: Build a `Room` in pyroomacoustics standards

First, a 3D mesh is loaded and separated into a list of vertices, and list of faces (triplets of vertex indices). In this example, materials are randomly asigned to each wall element.

In [ ]:
# Load the room shape in data/examples

room_mesh = tm.load_mesh(EXAMPLE_PATH / "dodecahedron.obj")
vertices = np.array(room_mesh.vertices) # shape (N_vertices, 3)
faces = np.array(room_mesh.faces) # shape (N_faces, 3) with each row containing the indices of the vertices that form a face

# We assign a random material to each face of the mesh, which will be used to create the walls of the pyroomacoustics room
material_assignment = np.random.choice(
    list(pra_material_library.keys()), size=len(faces)
)

Build `Wall` instances from each surface element and associated material, using the `wall_factory()` helper function from pyroomacoustics.

In [ ]:
walls = []
for i, face in enumerate(vertices[faces]):
    material_name = material_assignment[i]
    material = pra_material_library[material_name]
    face_corners = face.copy()
    wall = pra.wall_factory(
        corners=face_corners.T,
        absorption=material.absorption_coeffs,
        scattering=material.scattering_coeffs,
        name="wall_" + str(i),
    )
    walls.append(wall)

Build the `Room` from its walls

In [ ]:
room = pra.Room(walls=walls, fs=16000)

# check that the absorption coefficients of the first wall of the pyroomacoustics room are equal to the absorption coefficients of the material assigned to the corresponding face of the mesh
print("Absorption coefficients of the first wall:\n", room.walls[0].absorption)
print("Absorption coefficients of the assigned material:\n", np.array(material_dict[material_assignment[0]]))

#### Step 3: Build the `SWFTRoom`

When instanciating a `SWFTRoom`, the Statistical Wave Field Theory formulas are automatically applied to obtain the reverberation parameters (acoustic modes and colored reverberation time)

In [ ]:
swft_room_from_obj = ts.SWFTRoom(room=room)
swft_room_from_obj.room.plot()
plt.show()

In [ ]:
modes = swft_room.get_frequency_response_at_point([0., 0., 0.])
rt60s = swft_room.get_rt60_profile()

fig, axs = plt.subplots(2, 1, figsize=(12, 6))
axs[0].plot(swft_room.center_freqs, rt60s)
axs[0].set_title("RT60 profile of the room")
axs[0].set_xlabel("Frequency (Hz)")
axs[0].set_ylabel("RT60 (s)")

axs[1].plot(swft_room.center_freqs, modes)
axs[1].set_title("Acoustic modes at the center of the room")
axs[1].set_xlabel("Frequency (Hz)")
axs[1].set_ylabel("Magnitude")

plt.tight_layout() 

## 2 - Reverberator

In the Taylor-SWFT module, the `Reverberator` class is used to generate Room Impulse Responses from a `SWFTRoom` object and given source/receiver positions.

In [ ]:
# Instanciate the Taylor-SWFT reverberator with the created room
reverb = ts.Reverberator(room=swft_room)

# Define the receiver and source positions
receiver = np.array([0.0, 6.0, 1.5])
source = np.array([1.0, 1.0, 1.0])

# get the RIR at the receiver position for a source located at the defined source position
rir = reverb.get_rir_at_point(receiver, source_point=source, order=4)

In [ ]:
SR = reverb.sr
random_point = receiver
interpolated_rt = reverb.rt_profile
freqs = torch.linspace(0, SR / 2, len(interpolated_rt))
modes_ir = reverb.get_modes_at_point(random_point)
modes = torch.fft.rfft(modes_ir, n=len(interpolated_rt) * 2)[
    :-1
]

plt.rcParams.update({"font.size": 15})
plt.figure(figsize=(12, 6))
gs = GridSpec(2, 3, width_ratios=[1, 7, 0.1], height_ratios=[1, 2])
axs1 = plt.subplot(gs[1, 0])
axs2 = plt.subplot(gs[1, 1], sharey=axs1)
axs3 = plt.subplot(gs[0, 1], sharex=axs2)
axs4 = plt.subplot(gs[1, 2])

axs2.specgram(
    rir.cpu().numpy(), Fs=SR, NFFT=256, noverlap=128, cmap="plasma", vmin=-150
)
axs2.plot(interpolated_rt.cpu().numpy(), freqs, label="$RT_{60}$", color="cyan")
axs2.set_title("Spectrogram of the RIR")
axs2.set_xlabel("Time (s)")
plt.colorbar(axs2.images[0], label="Intensity (dB)", cax=axs4)
axs2.set_xlim(0, len(rir) // SR)
axs2.legend(loc="upper right")

axs1.plot(modes.abs().cpu().numpy(), freqs, color="magenta")
axs1.set_title("Initial energy")
axs1.set_ylabel("Frequency (Hz)")
axs1.set_xlabel("Magnitude")

axs3.plot(np.linspace(0, len(rir) / SR, len(rir)), rir.cpu().numpy())
axs3.set_title("Room Impulse Response")
axs3.set_xlabel("Time (s)")
axs3.set_ylabel("Amplitude")

plt.tight_layout()

## 3 - Real-Time Processing

First, load an audio that will be looped and passed in real-time through the reverberator

In [ ]:
from torchaudio import load, save
from torchaudio.functional import resample

input_audio, fs = load(EXAMPLE_PATH / "demo_sound.flac", normalize=True)
input_audio = resample(input_audio, orig_freq=fs, new_freq=SR)
input_audio = input_audio[0]  # load a mono audio file
#loop the audio file to make it longer than the rir
input_audio = input_audio.repeat(3)

plt.figure(figsize=(12, 3))
plt.specgram(
    input_audio.cpu().numpy(), Fs=SR, NFFT=256, noverlap=128, cmap="plasma", vmin=-150
)
plt.title("Spectrogram of the input audio")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

## High-level usage

The Taylor-SWFT module provides `TaylorSWFTRealTimeProcessor`, an API for real-time reverberation of an input audio, with an optional Graphic User Interface (GUI) for moving the source and receiver positions. Note that the full input signal needs to be provided to the API. To achieve real-time reverberation of a streamed input, please use the lower-level class `SWFTContext` (see the **Low-level usage** section below)

In [ ]:
taylor_swft_processor = ts.TaylorSWFTRealTimeProcessor(
    reverb=reverb,  # provide the reverberator object to the real-time processor
    input_audio=input_audio,  # input audio to be processed
    audio_buffer_dur=30e-3,  # duration of the audio buffer that will be processed at each iteration, in seconds.
    n_channels=1,  # number of channels of the input audio. Only mono audio is supported for now, so this should be set to 1.
    reflection_order=4,  # order of reflection considered for the ISM generated early echoes. High values will increase the processing time.
    overlap=True,  # whether to use overlapping buffers for processing the audio. Overlapping can help reduce artifacts at the boundaries of the processed audio buffers, but it will increase the computational load.
    device="cpu",  # device on which the processing will be performed.
    store_full_output=True,  # whether to store the full output of the processing for later analysis. Setting this to True will increase the memory usage.
    loop_input_audio=False,  # whether to cycle the input audio when the end of the audio is reached. If set to False, the processing will stop when the end of the audio is reached.
)

Before running the processor, it is needed to define a way to get the trajectories of the source and microphone. Two methods can be called:
- `set_trajectories_mode(dynamic_mic_pos, dynamic_source_pos)` : the processor will cycle through the provided microphone and source trajectories.
- `set_interactive_mode()` : this will enable a GUI, where the microphone and source trajectories are determined in real-time by the user. Only implemented for `SWFTRooms` generated with the `from_corners()` method. The additional python script `demo_interactive.py` illustrates this mode.

In [ ]:
sin_velocity = lambda t: 0.5 * (
    1 - np.cos(2 * np.pi * 0.1 * t)
)  # example of a velocity function for the source, which will make it move back and forth along a line at a speed of 0.5 m/s with a frequency of 0.1 Hz
T = sin_velocity(np.linspace(0, 20, 600))
source_trajectory = np.stack(
    (3 * T, np.ones_like(T), 1.5 * np.ones_like(T)), axis=1
)  # we create a trajectory that moves along the x-axis according to the defined velocity function, while staying at y=0 and z=0
receiver_trajectory = np.array(
    [2 * np.ones_like(T), 1 + 5 * T, 1.5 * np.ones_like(T)]
).T

taylor_swft_processor.set_trajectories_mode(
    receiver_trajectory, source_trajectory, display_graphics=False
)

**Run with disabled GUI and output audio stream** (faster computation time), then get the output reverberated audio

In [ ]:
tik = perf_counter()
output_audio = taylor_swft_processor.run()
tok = perf_counter()
print(f"Processing time: {tok - tik:.2f} seconds")
print(
    f"Average real time ratio (Processing time ÷ Audio duration): {(tok - tik) / (output_audio.shape[0] / SR):.2f}"
)

Audio(output_audio.cpu().numpy(), rate=SR)

In [ ]:
plt.figure(figsize=(12, 3))
plt.specgram(
    input_audio.cpu().numpy(), Fs=SR, NFFT=256, noverlap=128, cmap="plasma", vmin=-150
)
plt.title("Spectrogram of the input audio")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.figure(figsize=(12, 3))
plt.specgram(
    output_audio.cpu().numpy(), Fs=SR, NFFT=256, noverlap=128, cmap="plasma", vmin=-150
)
plt.title("Spectrogram of the output audio")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")

**Reset the processor's settings:** to change the settings of the real-time processor, it is needed to first reset its settings. Reset is also needed before re-running the processor.  

In [ ]:
taylor_swft_processor.reset_all()  # reset the processor to its initial state, allowing to run it again with different settings without having to create a new instance of the processor. This is useful for testing and experimentation purposes.

**Enable the audio output stream**

In [ ]:
taylor_swft_processor.set_output_stream()  # set the output stream of the processor, which will be used to play the processed audio in real-time.

**Enable the GUI in 'trajectories' mode**

In [ ]:
taylor_swft_processor.reset_all()
taylor_swft_processor.set_output_stream()
taylor_swft_processor.set_trajectories_mode(
    receiver_trajectory, source_trajectory, display_graphics=True
)
output = taylor_swft_processor.run()
Audio(output.cpu().numpy(), rate=SR)

**Enable GUI in interactive mode**

In [ ]:
taylor_swft_processor.reset_all()
taylor_swft_processor.set_output_stream()
taylor_swft_processor.set_interactive_mode()
output = taylor_swft_processor.run()
Audio(output.cpu().numpy(), rate=SR)

## Low-level usage

Any application that needs a low-level usage of the Taylor-SWFT algorithm should use the `SWFTContext` class. As for `TaylorSWFTRealTimeProcessor`, it should be instanciated with a predefined `Reverberator`.

In [ ]:
swft_context = ts.SWFTContext(
    rev=reverb,
    buffer_dur=30e-3,
    n_channels=1,
    reflection_order=2,
    device="cpu",
    overlap=True,
)

The method `process_next_buffer()` can be used to reverberate a buffer of input signal, given the source a microphone positions.

In [ ]:
receiver = np.array([0.0, 6.0, 1.5])
source = np.array([1.0, 1.0, 1.0])

input_buffer = input_audio[int(swft_context.buffer_size):2 * int(swft_context.buffer_size)].unsqueeze(-1)

tik = perf_counter()
output_buffer = swft_context.process_next_buffer(input_buffer, mic_pos=receiver, source_pos=source)
tok = perf_counter()
print(f"Processing time for one buffer: {tok - tik:.4f} seconds")
print(f"Buffer duration: {input_buffer.shape[0] / SR:.4f} seconds")
print(f"Real time ratio for one buffer: {(tok - tik) / (input_buffer.shape[0] / SR):.2f}")

Each time a new buffer is processed, the incoming buffer is piled over previous buffers in the inner input memory of `SWFTContext`. You can access to this memory through the attribute `input_circular_buffer`

In [ ]:
receiver = np.array([0.0, 6.0, 1.5])
source = np.array([1.0, 1.0, 1.0])

input_buffers = input_audio[:int(swft_context.buffer_size) * 4].reshape(-1, swft_context.buffer_size, swft_context.n_channels)
inner_memory = swft_context.input_circular_buffer

fig, axs = plt.subplots(len(input_buffers), 1, figsize=(6, 2 * len(input_buffers)))
for i, input_buffer in enumerate(input_buffers):
    print(f"Processing buffer {i + 1}/{len(input_buffers)}")
    tik = perf_counter()
    output_buffer = swft_context.process_next_buffer(input_buffer, mic_pos=receiver, source_pos=source)
    tok = perf_counter()
    print(f"Processing time for buffer {i + 1}: {tok - tik:.4f} seconds")
    print(f"Real time ratio for buffer {i + 1}: {(tok - tik) / (input_buffer.shape[0] / SR):.2f}")
    current_memory_state = inner_memory.get().flatten().cpu().numpy()
    print(current_memory_state.shape)
    axs[i].plot(current_memory_state)
    axs[i].set_title(f"Input memory state after processing buffer {i + 1}")
    axs[i].set_xlabel("Sample index")
    axs[i].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()


Similarly, output buffers are stacked in queue, with a first in first out behavior. The oldest memorized output buffer can be popped out of the queue the the method `get_next_output_buffer()`

In [ ]:
fig, axs = plt.subplots(len(swft_context.output_queue), 1, figsize=(6, 2 * len(input_buffers)), sharey=True)
output_buffers = []
for i in range(len(swft_context.output_queue)):
    output_buffer = swft_context.get_next_output_buffer()
    output_buffers.append(output_buffer)
    print(f'Popped output buffer {i + 1}/{len(swft_context.output_queue)}. output_queue size is now {len(swft_context.output_queue)}')

    full_output = np.stack(output_buffers, axis=0).flatten()
    axs[i].plot(full_output)
    axs[i].set_title(f"Output signal at iteration {i + 1}")
    axs[i].set_xlabel("Sample index")
    axs[i].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()


Finally, call `reset()` to clear the inner memory and reset all internal parameters.

In [ ]:
swft_context.reset()
print("Output queue size after reset:", len(swft_context.output_queue))
print("Input memory maximum amplitude after reset:", swft_context.input_circular_buffer.get().flatten().abs().max())